# 🛠️ 02 — Nettoyage & Feature Engineering
**Objectif** : Préparer les données pour la modélisation (nettoyage, encodage, nouvelles features).

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv(r'C:\Users\imene\churn-prediction\data\WA_Fn-UseC_-Telco-Customer-Churn.csv')
print(f"Dataset brut : {df.shape}")

## 1. Nettoyage

In [ ]:
df2 = df.copy()

# Suppression customerID (identifiant non prédictif)
df2.drop('customerID', axis=1, inplace=True)

# TotalCharges : conversion et imputation
df2['TotalCharges'] = pd.to_numeric(df2['TotalCharges'], errors='coerce')
n_missing = df2['TotalCharges'].isnull().sum()
df2['TotalCharges'] = df2['TotalCharges'].fillna(df2['TotalCharges'].median())
print(f"TotalCharges — {n_missing} valeurs manquantes imputées par la médiane")

# Variable cible
df2['Churn'] = (df2['Churn'] == 'Yes').astype(int)
print(f"Taux de churn : {df2['Churn'].mean()*100:.1f}%")

## 2. Feature Engineering

In [ ]:
# Charges mensuelles moyennes (évite la colinéarité TotalCharges/tenure)
df2['ChargesParMois'] = df2['TotalCharges'] / (df2['tenure'] + 1)

# Nombre de services souscrits
services = ['PhoneService', 'MultipleLines', 'OnlineSecurity',
            'OnlineBackup', 'DeviceProtection', 'TechSupport',
            'StreamingTV', 'StreamingMovies']
df2['NbServices'] = sum((df2[s] == 'Yes').astype(int) for s in services)

# Flags métier
df2['ClientFidele'] = (df2['tenure'] > 24).astype(int)
df2['ContratCourt'] = (df2['Contract'] == 'Month-to-month').astype(int)

print("Features créées :")
print(df2[['ChargesParMois', 'NbServices', 'ClientFidele', 'ContratCourt']].describe())

## 3. Encodage

In [ ]:
cat_cols = df2.select_dtypes(include='object').columns.tolist()
print(f"Colonnes catégorielles à encoder : {cat_cols}")

df_enc = pd.get_dummies(df2, columns=cat_cols, drop_first=True)
df_enc = df_enc.fillna(0)

print(f"\nDataset encodé : {df_enc.shape[1]} colonnes")
print(f"NaN résiduels : {df_enc.isnull().sum().sum()}")

## 4. Export

In [ ]:
df2.to_csv(r'C:\Users\imene\churn-prediction\data\telco_clean.csv', index=False)
df_enc.to_csv(r'C:\Users\imene\churn-prediction\data\telco_encoded.csv', index=False)

print("✅ telco_clean.csv exporté")
print("✅ telco_encoded.csv exporté")